In [2]:
# read our shakespeare dataset to inspect it
with open("input.txt", "r") as f:
    text = f.read()
print(f"length of characters in dataset: {len(text)}")


length of characters in dataset: 1115394


In [3]:
#lets look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
# get and sort the set of characters that occur in this text, put it in a list
chars = sorted(list(set(text))) #0 will be newline character, 1 will be space character, 2 will be "a" and so on
vocab_size = len(chars)
print("all the unique characters:", "".join(chars))
print(f"vocab size: {vocab_size}")

all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65


In [5]:
#create a mapping from characters to integers(character-level tokenization)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i: ch for i,ch in enumerate(chars)}  
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers (lambda = function) 
decode = lambda l: "".join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
#now lets encode the entire dataset and store it in a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long) # torch.long = 64-bit integer (not float)
print(data.shape, data.dtype)
print(data[:1000]) # the first 1000 characters we looked at ealier, encoded as integers (input to GPT)

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [7]:
#split the data into train and validation/test sets
n = int(0.9*len(data)) # first 90% will be train, rest will be validation/test
train_data = data[:n]
val_data = data[n:] #keep the validation data to the side for now, use it later to evaluate our model after training

# no test set because this is a generative model — evaluate by looking at the generated text qualitatively, 
# and by tracking val loss.

In [8]:
#block_size = context length for predictions (num characters model sees  when predicting the next character)
#batch_size = num of independent sequences we process in parallel (for training efficiency)

block_size = 8
train_data[:block_size+1] 

# we want to use the first 8 characters to predict the 9th character, 
#so we need block_size+1 characters of data

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size] # the first 8 characters will be the input to the model
y = train_data[1:block_size+1] #the next 8 characters (y is offset by 1) will be the target (the model should learn to predict these from the input)
for t in range(block_size): 
    context = x[:t+1] # context grows each step, from just the first character at t=0, to the full 8 characters at t=7 (range starts at 0)
    target = y[t] #target is the single next character that follows the context
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1337) # for reproducibility
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # randomly select batch_size starting points for the sequences
    x = torch.stack([data[i:i+block_size] for i in ix]) # take all the 1D tensors in ix, and stack them up as rows(a tensor of shape (batch_size, block_size))
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # the targets are the same sequences but shifted by one character
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape) # (batch_size, block_size) = (4, 8) = 4 sequences/rows of 8 characters/columns each, as input
print(xb) # the above4 sequences of 8 characters each, encoded as integers (input to GPT)
print("\ntargets:")
print(yb.shape) 
print(yb) # same 4 sequences of 8 characters each, but shifted by one character, encoded as integers (the target output that GPT should learn to predict)

# yb only comes into transformer at the end to create/compute the loss function(gives correct answer for every single position in x)

print ("---")
for b in range(batch_size): 
    for t in range(block_size):
        context = xb[b, :t+1] # the context is growing with each step, from just the first character at t=0, to the full 8 characters at t=7
        target = yb[b, t] # the target is the single next character that follows the context
        print(f"when input is {context.tolist()} the target: {target.item()}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [11]:
print(xb) # input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
# bigram model from pytorch to establish baseline and eval loop
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # input is an integer (token), output is a vector of logits of size vocab_size (logits for the next token)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensors of integers
        logits = self.token_embedding_table(idx) # (B,T,C) where B = block, T = batch, C = channels 
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step (bc its bigram model)
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

#generate some text from the model(gna be garbage because model is not trained yet)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())) 

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [13]:
# now lets train the bigram model
# using the AdamW optimizer, which is a variant of stochastic gradient descent suited for training transformer models. 
# this optimizer object will update the parameters of the model based on the computed gradients during training

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #learning rate of 1e-3, on m.parameters() = all the parameters of the model that we want to optimize

In [14]:
#training loop
batch_size = 32

for steps in range(10000):

    # sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # set gradients to zero before backpropagation
    loss.backward() # backpropagation to compute gradients of the loss with respect to the model parameters
    optimizer.step() # update the model parameters using the computed gradients

print(loss.item()) # print the loss at the end of training, should be around 2.0 (cross entropy loss of random predictions) or lower if the model has learned something


2.382369041442871


##  Bigram model gets loss ~2.5.  GPT will get ~1.48

In [15]:
#used trained bigram model to generate some text (not good cuz it only has 1 char of context)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist())) 


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecor


## the math trick behind self-attention

In [ ]:
# toy example to get used to the operatio
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch size, time steps(8 tokens in a batch), channels (dimensions of the each token's embeddings)
x = torch.randn(B, T, C) # random input tensor of shape (B, T, C)
print(x.shape) # should be (4, 8, 2)


torch.Size([4, 8, 2])


### Self attention: we want the token at the nth position(theres 8 token positions) to talk to only all the tokens that came before it.
 Version 1 — Averaging with For Loops (Weakest Aggregation)

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # x bag of words = the mean of all the previous word embeddings in the sequence, for each position in the sequence (the "bag of words" representation of the context)
for b in range(B): # iterate over the batch dimension
    for t in range(T): # iterate over the time steps(each token)
        xprev = x[b,:t+1] # (t,C). # [t+1, C] — all tokens up to and including t
        xbow[b,t] = torch.mean(xprev, 0) #average them 

x[0] # the original input tensor for the first sequence in the batch (8 tokens, each with 2 channels)

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [ ]:
xbow[0] # each token is represented by the mean of all the previous tokens in the sequence (including itself)

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

### Version 2 — Matrix Multiply as Weighted Aggregation (The Trick)
The averaging above can be expressed as a matrix multiply. Create a lower-triangular matrix of weights:

In [ ]:
wei = torch.tril(torch.ones(T, T))      # lower triangular: 1s on and below diagonal
wei = wei / wei.sum(1, keepdim=True)    # normalize each row to sum to 1

# Now: [T, T] @ [B, T, C] — but we need to batch this:
xbow2 = wei @ x   # [T, T] @ [T, C] = [T, C], batched: [B, T, T] @ [B, T, C] = [B, T, C]
torch.allclose(xbow, xbow3) # check that the two methods of computing xbow give the same result (they should)

### What the lower triangular matrix means:

```text
Position 0: [1,   0,   0,   0,   ...]  <- only sees itself
Position 1: [0.5, 0.5, 0,   0,   ...]  <- averages positions 0 and 1
Position 2: [0.33,0.33,0.33,0,   ...]  <- averages positions 0, 1, 2
Position 3: [0.25,0.25,0.25,0.25,...]  <- averages positions 0-3

The upper triangle is zero — future tokens are masked out. A token at position 3 cannot see tokens at positions 4, 5, 6... This is called causal masking and is fundamental to the decoder architecture.

🔑 Lower triangular masking = "only look at the past." This is what makes the model autoregressive — it can only condition on what came before, not what comes after. This is why it can generate text one token at a time.

### version 3: add softmax

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T)) # lower triangular: 1s on and below diagonal, 0s above diagonal
wei = torch.zeros((T,T)) #begins as a matrix of zeros
wei = wei.masked_fill(tril == 0, float('-inf')) # set the upper triangular part (where tril == 0) to -infinity, so that after softmax these positions will have zero weight
wei = F.softmax(wei, dim=-1) # apply softmax to get the weights, which will be 0 for the upper triangular part and normalized for the lower triangular part
xbow3 = wei @ x
torch.allclose(xbow, xbow3) # check that the two methods of computing xbow give the same result (they should)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

masked_fill(tril == 0, float('-inf')) — set upper triangle to -∞. 

After softmax, e^(-∞) = 0, so those positions contribute nothing.


This produces the same uniform average as before (since wei starts as zeros → equal weights after softmax). But now the structure is ready for 
### Version 4: make wei data-dependent instead of uniform.

In [ ]:
# rn, each token in the context block(row) has uniform (average) attention to all previous tokens  
wei 

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

### self attention
- instead of averaging uniformly, each token should pay different amounts of attention to different past tokens, based on what it's "looking for" and what past tokens "have to offer." 

### every single token at each position will emit 2 vectors. 
- query : what am i looking for?
- key : what do i contain?


### dot product between keys and queries to compute values. 
- each token's query dot products with each other token in the sequence's key's. 
- The dot product of a Query with all Keys gives an affinity score: "how relevant is this past token to me?"
- After masking and softmax, those scores become weights
- Each token then aggregates all past tokens' Values using those weights

### implement one head of self-attention

In [32]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key   = nn.Linear(C, head_size, bias=False)  # "what do I have?"
query = nn.Linear(C, head_size, bias=False)  # "what am I looking for?"
value = nn.Linear(C, head_size, bias=False)  # "what do I actually give out?"

k = key(x)    # [B, T, head_size]
q = query(x)  # [B, T, head_size]

# Compute affinities: how much does each query "match" each key?
wei = q @ k.transpose(-2, -1)  # [B, T, head_size] @ [B, head_size, T] = [B, T, T]

# Scale (explained below)
wei = wei * head_size**-0.5

# Causal mask
tril = torch.tril(torch.ones(T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))

# Normalize weights
wei  = F.softmax(wei, dim=-1)  # [B, T, T]

# Apply to values
v   = value(x)    # [B, T, head_size]
out = wei @ v     # [B, T, T] @ [B, T, head_size] = [B, T, head_size]

The Q/K dot product is a learned similarity. Unlike the uniform average, the model learns which past tokens are relevant for each position. A word looking for a verb will have a query that scores high dot products against key vectors of verb-like tokens. The whole thing is differentiable and learned end-to-end.

### Scaling
Why divide by sqrt(head_size)?
If head_size is large, the dot products q @ k^T become large in magnitude → softmax gets very "peaky" → gradients vanish. Dividing by √head_size keeps the variance of the dot products at ~1 regardless of head_size. This is "scaled dot-product attention."

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.

- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to **positionally encode tokens**.

- Each example across batch dimension is processed completely independently and never "talk" to each other
- In an **"encoder" attention block** just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a **"decoder" attention block** because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. 

# Section 6 — Building the Full Architecture, Layer by Layer
### Single Attention Head as a Class:

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out
    
   # self.register_buffer registers a tensor as a non-parameter buffer. The tril mask doesn't need gradients and shouldn't be updated, 
   # but it needs to be part of the model (saved with it, moved to GPU with it). register_buffer does exactly that.

### Multi-Head Attention
Instead of one attention head, run several in parallel and concatenate the results:

Why multiple heads? 
- Each head can learn to attend to different aspects simultaneously — one head might track subject-verb agreement, another might track coreference.
- num_heads * head_size == n_embd so the total dimension is preserved.
- The proj (projection) layer maps the concatenated output back to n_embd. This is a learned linear combination of all heads' outputs.

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # create a list of Head modules, one for each head of attention
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout) 

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the output of all the heads together, along the channels dimension (the last dimension)
        out = self.dropout(self.proj(out)) 
        return out

### FeedForward Block
After attention (communication between tokens), each token processes its own updated representation **independently**:

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


- The 4 * n_embd expansion factor comes from the original "Attention Is All You Need" paper. Note: this uses ReLU not tanh — GPT uses ReLU in the feedforward blocks.

🔑 Attention = communication. FeedForward = computation. 

- The attention layer lets tokens look at each other and gather context. 
- The feedforward layer then lets each token "think about" what it gathered, independently. 
- Together, one repetition of [Attention → FeedForward] is **one Transformer block.**

### Residual Connections

What is a residual connection?
- Instead of x = layer(x), you do x = x + layer(x). The layer only needs to learn the difference (residual) from the input, not the full transformation. 
- The original x is passed through a "skip connection" that bypasses the layer.

Why residuals are critical for deep networks:

- During backpropagation, gradients flow back through + unmodified. So even if a layer has near-zero gradients, the gradient still flows straight through the skip connection to earlier layers. This is what allows training networks with 12, 24, or 96 layers without gradient vanishing.

🔑 Think of residuals as a "gradient highway." 
- Deep layers add refinements on top of an existing representation. Early layers don't have to wait for gradients to pass through 12 nonlinearities — they get a direct copy via the skip connections.

### Layer Normalization (LayerNorm)
Layer normalization **stabilizes training** by normalizing the activations across the feature dimension (within one example) 
-  normalizes over the feature dimension (within one example), where BatchNorm normalized over the batch dimension (across examples)
- ^ so: each token normalized independently
- LayerNorm is used for Transformers, where BatchNorm was used for MLPs and CNN's                                                                                                          
  
Why it matters in transformers:

  1. Prevents gradient problems - Without it, activations can grow or shrink exponentially as you stack layers. LayerNorm keeps them in a
  well-behaved range, so gradients flow cleanly during backprop.
  2. Enables deep networks - You can stack many transformer blocks (12, 24, 96...) without training collapsing. It's what makes depth
  practical.
  3. Stabilizes self-attention - The dot products in attention (Q @ K.T) scale with embedding dimension, so values can get large. LayerNorm
   before attention (pre-norm, used in modern GPTs) keeps inputs bounded.


Pre-norm vs. post-norm: 
- The original "Attention Is All You Need" paper applied LayerNorm after the attention/FFN (post-norm). 
- we apply it before (pre-norm): x + sa(ln(x)). 
- Pre-norm is the modern convention — it creates a cleaner gradient highway through the residual stream and trains more stably.

### Dropout

self.dropout = nn.Dropout(dropout)  # dropout = 0.2 typically
- During training, randomly zeros out 20% of activations each forward pass. Each training step sees a slightly different sub-network — this forces redundancy and prevents co-adaptation of neurons.
- At inference (model.eval()), dropout is disabled automatically — all neurons are active. You don't need to do anything special; nn.Module handles the training flag.

# FULL transformer block architecture
a transformer block has 2 parts : self-attention and feedforward. (u layernormalize before each step)

1) layernorm (normalize) the input 'x', then compute self-attention, then add the input 'x' to the output of the self attention
2) layer normalize again before feedforward network (MLP), then feed-forward, then add input 'x' to output of FFWD (residual connection)


In [ ]:
class Block(nn.Module):
    """ FULL Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) # self attention
        self.ffwd = FeedFoward(n_embd) # feed forward network (MLP)
        self.ln1 = nn.LayerNorm(n_embd) # layer normalization before self attention, to stabilize training (normalizes the inputs to have mean 0 and variance 1 across the features dimension)
        self.ln2 = nn.LayerNorm(n_embd) # layer normalization before feed forward network

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # self attention with residual connection (add the input x to the output of the self attention, to help with gradient flow and training stability)
        x = x + self.ffwd(self.ln2(x)) # feed forward network with residual connection
        return x


# FULL GPT MODEL


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd) # each token gets mapped to a vector of size n_embd (the embedding dimension)
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position in the context block gets mapped to a vector of size n_embd 
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # a stack of n_layer blocks, each containing multi-head self attention and feed forward network
        self.ln_f   = nn.LayerNorm(n_embd)  # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size) # the linear layer that maps from the final embedding dimension to the vocabulary size, to produce logits for the next token prediction

    def forward(self, idx, targets=None):
        B, T = idx.shape # batch size, time steps
        tok_emb = self.token_embedding_table(idx)  # [B, T, n_embd]
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # [T, n_embd]
        x = tok_emb + pos_emb   # [B, T, n_embd] ;input is the sum of the token and position embeddings (so model knows which position in the sequence each token is at)
        x = self.blocks(x) # pass through the stack of transformer blocks
        x = self.ln_f(x) # apply layer normalization to the output of the last transformer block
        logits = self.lm_head(x) # [B, T, vocab_size] ; output logits for the next token prediction for each position in the input sequence                                           

        if targets is None: 
            loss = None
        else:
            B, T, C = logits.shape 
            logits  = logits.view(B*T, C) # reshape logits to be a 2D tensor of shape (B*T, C), where each row corresponds to the logits for the next token prediction for a single position in the input sequence
            targets = targets.view(B*T) # reshape targets to be a 1D tensor of shape (B*T), where each element is the index of the correct next token for the corresponding position in the input sequence
            loss    = F.cross_entropy(logits, targets) # compute the cross-entropy loss between the predicted logits and the target tokens
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # crop to block_size
            logits, _ = self(idx_cond) # get the predictions
            logits = logits[:, -1, :] # focus only on the last time step's logits, which correspond to the next token prediction for the current context
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution to get the next token index
            idx = torch.cat((idx, idx_next), dim=1) # append sampled index to the running sequence
        return idx

### Positional embeddings: 
- just like token embeddings, but for position instead of token identity. torch.arange(T) creates [0, 1, 2, ..., T-1] — the position indices. 
- The model learns a separate embedding vector for each position. Adding tok_emb + pos_emb gives each token both its identity and its position.
### The idx[:, -block_size:] crop in generate: 
- as generation continues, the context grows. But the model can only handle block_size tokens at once — crop to the last block_size tokens.

# hyperparemeters 
### Small (trains quickly on CPU, for demonstration)
- batch_size  = 32
- block_size  = 8
- n_embd      = 32
- n_head      = 4
- n_layer     = 3
- dropout     = 0.0
- learning_rate = 1e-3
- max_iters   = 5000

### Scaled (requires GPU, ~1M params)
- batch_size  = 64
- block_size  = 256
- n_embd      = 384
- n_head      = 6
- n_layer     = 6
- dropout     = 0.2
- learning_rate = 3e-4
- max_iters   = 5000

The training loop uses Adam optimizer (not plain SGD):

- pythonoptimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

^ AdamW is the standard for transformers. It adapts the learning rate per parameter and includes weight decay. it converges much faster than SGD on transformer architectures.

Val loss progression as pieces are added:

- Bigram baseline: ~2.5
- self-attention: ~2.4
- feedforward: ~2.2
- residuals + LayerNorm: ~2.1
- scale up: ~1.48